# 🧥 MeshVTON: Full 3D Virtual Try-On Training

With this notebook, you can run the entire pipeline:

1. GPU and library setup
2. Mounting data from Google Drive
3. SMPL-X body parameter extraction
4. 3D garment rendering
5. Model training

Requirements: GPU runtime (T4/A100/H100)

---
## 1️⃣ GPU Control

In [1]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU not found! Runtime → Change runtime type → Select GPU')

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


---
## 2️⃣ Clone the project

In [2]:
import os

# === CHANGE THIS: Your own GitHub repo URL ===
REPO_URL = 'https://github.com/SerhanTelatar/MeshVTON.git'
PROJECT_DIR = '/content/MeshVTON'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print('✅ Repo cloned')
else:
    !cd {PROJECT_DIR} && git pull
    print('✅ Repo updated')

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

Already up to date.
✅ Repo updated
Working directory: /content/MeshVTON


---
## 3️⃣ Install the Libraries

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:

# Drive'dan yükle — derleme yok
from google.colab import drive
drive.mount('/content/drive')

!pip install -q fvcore iopath
!pip install /content/drive/MyDrive/wheels/pytorch3d*.whl

# Core dependencies — FIXED VERSIONS for IDM-VTON compatibility
!pip install diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 huggingface_hub==0.20.3 peft==0.7.1 -q
!pip install -q omegaconf opencv-python-headless pillow scipy
!pip install -q lpips einops timm

# 3D Pipeline dependencies
!pip install -q smplx trimesh pyrender

# PyTorch3D
try:
    import pytorch3d
    print(f'✅ PyTorch3D: {pytorch3d.__version__}')
except:
    !pip install -q "git+https://github.com/facebookresearch/pytorch3d.git"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing /content/drive/MyDrive/wheels/pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl
pytorch3d is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
✅ PyTorch3D: 0.7.9


---
## 4️⃣ Connect Google Drive & Turn on Data

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# === CHANGE THIS: Your folder path in Drive ===
DRIVE_DATA = '/content/drive/MyDrive/MeshVTON'

import os
print('Drive content:')
for f in os.listdir(DRIVE_DATA):
    size = os.path.getsize(os.path.join(DRIVE_DATA, f)) / 1e6
    print(f'  {f} ({size:.1f} MB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive content:
  test_pairs.csv (0.1 MB)
  val_pairs.csv (0.1 MB)
  train_pairs.csv (1.1 MB)
  pretrained.zip (226.8 MB)
  images.zip (1817.5 MB)
  poses.zip (169.1 MB)
  segments.zip (98.5 MB)
  densepose.zip (645.0 MB)
  agnostic.zip (429.5 MB)
  garments_3d.zip (5363.9 MB)
  checkpoints (0.0 MB)
  smplx_params.zip (9.9 MB)
  pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl (63.5 MB)
  MeshVTON_Train.ipynb (0.1 MB)
  renders_3d.zip (151.4 MB)
  normal_maps.zip (276.9 MB)
  depth_maps.zip (85.8 MB)


In [6]:
import zipfile
import shutil
from pathlib import Path

PROJECT = Path('/content/MeshVTON')
DRIVE = Path(DRIVE_DATA)

def extract_zip(zip_name, target_dir):
    """Extract zip from Drive (skips if not present)."""
    zip_path = DRIVE / zip_name
    if not zip_path.exists():
        print(f'  ⚠️ {zip_name} not found, skipping')
        return
    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    print(f'  📦 Extracting {zip_name} → {target_dir}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target)
    print(f'  ✅ {zip_name} completed')

for d in ['data/raw/images', 'data/processed/poses', 'data/processed/segments',
          'data/processed/densepose', 'data/processed/agnostic',
          'data/processed/smplx_params', 'data/processed/renders_3d',
          'data/processed/normal_maps', 'data/processed/depth_maps',
          'data/garments_3d', 'checkpoints/pretrained']:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)

print('📂 Extracting zip files...')
print()

extract_zip('images.zip', PROJECT / 'data/raw/images')

extract_zip('poses.zip', PROJECT / 'data/processed')
extract_zip('segments.zip', PROJECT / 'data/processed')
extract_zip('densepose.zip', PROJECT / 'data/processed')
extract_zip('agnostic.zip', PROJECT / 'data/processed')

extract_zip('garments_3d.zip', PROJECT / 'data/garments_3d')

extract_zip('pretrained.zip', PROJECT / 'checkpoints')

# --- CACHED preprocessing outputs: extract instead of recomputing ---
# (saved to Drive by the final "Save to Drive" cell on a previous run)
extract_zip('smplx_params.zip', PROJECT / 'data/processed/smplx_params')
extract_zip('renders_3d.zip',  PROJECT / 'data/processed/renders_3d')
extract_zip('normal_maps.zip', PROJECT / 'data/processed/normal_maps')
extract_zip('depth_maps.zip',  PROJECT / 'data/processed/depth_maps')

for csv_name in ['train_pairs.csv', 'val_pairs.csv', 'test_pairs.csv']:
    s = DRIVE / csv_name
    if s.exists():
        shutil.copy2(s, PROJECT / 'data/raw' / csv_name)

print()
print('✅ All data loaded!')

📂 Extracting zip files...

  📦 Extracting images.zip → /content/MeshVTON/data/raw/images...
  ✅ images.zip completed
  📦 Extracting poses.zip → /content/MeshVTON/data/processed...
  ✅ poses.zip completed
  📦 Extracting segments.zip → /content/MeshVTON/data/processed...
  ✅ segments.zip completed
  📦 Extracting densepose.zip → /content/MeshVTON/data/processed...
  ✅ densepose.zip completed
  📦 Extracting agnostic.zip → /content/MeshVTON/data/processed...
  ✅ agnostic.zip completed
  📦 Extracting garments_3d.zip → /content/MeshVTON/data/garments_3d...
  ✅ garments_3d.zip completed
  📦 Extracting pretrained.zip → /content/MeshVTON/checkpoints...
  ✅ pretrained.zip completed
  📦 Extracting smplx_params.zip → /content/MeshVTON/data/processed/smplx_params...
  ✅ smplx_params.zip completed
  📦 Extracting renders_3d.zip → /content/MeshVTON/data/processed/renders_3d...
  ✅ renders_3d.zip completed
  📦 Extracting normal_maps.zip → /content/MeshVTON/data/processed/normal_maps...
  ✅ normal_maps.z

In [7]:
"""Fix data paths and verify — SINGLE CELL"""
import shutil, subprocess, os
from pathlib import Path
P = Path('/content/MeshVTON')
# ============================================================
# 1. Fix nested extraction paths
# ============================================================
nested_img = P / 'data/raw/images/images'
if nested_img.exists():
    for f in nested_img.glob('*'):
        shutil.move(str(f), str(P / 'data/raw/images' / f.name))
    shutil.rmtree(nested_img, ignore_errors=True)
    print('🔧 Person images fixed')
nested_g = P / 'data/garments_3d/garments_3d'
if nested_g.exists():
    for d in nested_g.iterdir():
        dest = P / 'data/garments_3d' / d.name
        if not dest.exists():
            shutil.move(str(d), str(dest))
    shutil.rmtree(nested_g, ignore_errors=True)
    print('🔧 3D garments fixed')
smplx_target = P / 'checkpoints/pretrained/smplx'
smplx_target.mkdir(parents=True, exist_ok=True)
result = subprocess.run(['find', str(P / 'checkpoints'), '-name', '*.npz'],
                       capture_output=True, text=True)
for line in result.stdout.strip().split('\n'):
    if line.strip():
        src = Path(line.strip())
        if src.exists() and 'NEUTRAL_2020' not in src.name:
            dest = smplx_target / 'SMPLX_NEUTRAL.npz'
            if src.resolve() != dest.resolve():
                shutil.copy2(str(src), str(dest))
                print(f'🔧 SMPL-X → {dest}')
        elif src.exists() and 'NEUTRAL_2020' in src.name:
            dest = smplx_target / 'SMPLX_NEUTRAL_2020.npz'
            if src.resolve() != dest.resolve():
                shutil.copy2(str(src), str(dest))
vposer_target = P / 'checkpoints/pretrained/vposer'
vposer_target.mkdir(parents=True, exist_ok=True)
# ============================================================
# 2. Final Verification
# ============================================================
print('\n📊 Final Data Check')
print('=' * 45)
checks = {
    'Person Images':      len(list((P/'data/raw/images').glob('*.jpg'))),
    'Poses':              len(list((P/'data/processed/poses').glob('*'))),
    'Segmentation':       len(list((P/'data/processed/segments').glob('*'))),
    'DensePose':          len(list((P/'data/processed/densepose').glob('*'))),
    'Agnostic':           len(list((P/'data/processed/agnostic').glob('*'))),
    '3D Garment (upper)': len(list((P/'data/garments_3d/upper_body').glob('*'))) if (P/'data/garments_3d/upper_body').exists() else 0,
    '3D Garment (lower)': len(list((P/'data/garments_3d/lower_body').glob('*'))) if (P/'data/garments_3d/lower_body').exists() else 0,
    '3D Garment (dress)': len(list((P/'data/garments_3d/dresses').glob('*'))) if (P/'data/garments_3d/dresses').exists() else 0,
    '3D Garment (outer)': len(list((P/'data/garments_3d/outerwear').glob('*'))) if (P/'data/garments_3d/outerwear').exists() else 0,
    'SMPL-X Model':       (P/'checkpoints/pretrained/smplx/SMPLX_NEUTRAL.npz').exists(),
    'train_pairs.csv':    (P/'data/raw/train_pairs.csv').exists(),
}
all_ok = True
for name, val in checks.items():
    ok = '✅' if val else '❌'
    if not val: all_ok = False
    print(f'  {ok} {name}: {val}')
print('=' * 45)
print('🎉 ALL READY!' if all_ok else '⚠️ Some data is missing')

🔧 Person images fixed
🔧 3D garments fixed
🔧 SMPL-X → /content/MeshVTON/checkpoints/pretrained/smplx/SMPLX_NEUTRAL.npz

📊 Final Data Check
  ✅ Person Images: 13679
  ✅ Poses: 11648
  ✅ Segmentation: 11647
  ✅ DensePose: 11647
  ✅ Agnostic: 11647
  ✅ 3D Garment (upper): 218
  ✅ 3D Garment (lower): 218
  ✅ 3D Garment (dress): 182
  ✅ 3D Garment (outer): 200
  ✅ SMPL-X Model: True
  ✅ train_pairs.csv: True
🎉 ALL READY!


---
## 5️⃣ SMPL-X Body Parameter Extraction

Her kişi görüntüsünden 3D beden parametrelerini (şekil, poz) çıkarir.

In [8]:
import os
os.chdir('/content/MeshVTON')
# __init__.py dosyalarını oluştur (Python'un modülleri bulması için)
from pathlib import Path
for d in ['src', 'src/data', 'src/data/preprocessing', 'src/modules', 'src/models']:
    init = Path(d) / '__init__.py'
    init.parent.mkdir(parents=True, exist_ok=True)
    if not init.exists():
        init.touch()
        print(f'Created {init}')
print('✅ Done — şimdi SMPL-X hücresini tekrar çalıştır')

✅ Done — şimdi SMPL-X hücresini tekrar çalıştır


In [ ]:
# SMPL-X + 4D-Humans (HMR2.0) — gerçek poz için TÜM model dosyaları
!pip install -q smplx
import os, glob, shutil

# 1) SMPL-X neutral (bizim renderer/draper) — Drive'dan
SMPLX_DIR = '/content/MeshVTON/checkpoints/pretrained/smplx'
os.makedirs(SMPLX_DIR, exist_ok=True)
for s in glob.glob('/content/drive/MyDrive/MeshVTON/smplx/SMPLX_NEUTRAL.*'):
    shutil.copy(s, SMPLX_DIR)
assert glob.glob(f'{SMPLX_DIR}/SMPLX_NEUTRAL.*'), \
    "SMPLX_NEUTRAL.npz yok → smpl-x.is.tue.mpg.de'den indir, Drive/MeshVTON/smplx/"

# 2) 4D-Humans kurulumu
try:
    import hmr2  # noqa
except Exception:
    get_ipython().system('pip install -q git+https://github.com/shubham-goel/4D-Humans.git')

# 3) HMR2.0 ağırlıkları (checkpoint + config) — bir kez iner (~3.5 GB)
from hmr2.models import download_models
from hmr2.configs import CACHE_DIR_4DHUMANS
download_models(CACHE_DIR_4DHUMANS)

# 4) SMPL neutral (HMR2'nin gövde modeli) — smplify.is.tue.mpg.de → Drive'dan
smpl_dir = f"{CACHE_DIR_4DHUMANS}/data/smpl"; os.makedirs(smpl_dir, exist_ok=True)
cands = glob.glob('/content/drive/MyDrive/MeshVTON/smpl/*neutral*lbs*.pkl') \
      + glob.glob('/content/drive/MyDrive/MeshVTON/smpl/SMPL_NEUTRAL.pkl')
assert cands, "SMPL neutral pkl yok → smplify.is.tue.mpg.de (SMPLIFY_CODE_V2.zip), Drive/MeshVTON/smpl/"
shutil.copy(cands[0], f"{smpl_dir}/SMPL_NEUTRAL.pkl")
print('✅ SMPL-X + HMR2.0 + SMPL neutral hazır')

In [ ]:
import sys, os, shutil
os.chdir('/content/MeshVTON'); sys.path.insert(0, '/content/MeshVTON')
!git pull
from pathlib import Path

# ⚠️ Eski smplx_params/renderlar EĞİTİLMEMİŞ poz (yanlış yön) + GRİ render ile
# üretilmişti. Gerçek poz + dokulu render için bir kez yeniden üret.
# İlk doğru üretimden sonra FORCE_REGEN=False yapıp tekrar üretmeyi atlayabilirsin.
FORCE_REGEN = True

smplx_out = Path('data/processed/smplx_params')
if FORCE_REGEN and smplx_out.exists():
    shutil.rmtree(smplx_out); print('🗑️ eski smplx_params silindi (yeniden üretilecek)')

existing = list(smplx_out.glob('*.npz')) if smplx_out.exists() else []
if existing:
    print(f'⏩ SMPL-X params mevcut ({len(existing)}) — atlanıyor')
else:
    from src.data.preprocessing.extract_smplx import extract_smplx
    extract_smplx(
        image_dir='data/raw/images',
        output_dir='data/processed/smplx_params',
        model_dir='checkpoints/pretrained',
        device='cuda',
        save_mesh=True,
        mesh_dir='data/processed/smplx_meshes',
        regressor='hmr2',   # gerçek poz (çıkarım notebook'u ile AYNI backend)
    )

---
## 6️⃣ 3D Clothing Rendering

CLOTH3D mesh'lerini SMPL-X beden modellerine giydirip 2D'ye render eder.

In [11]:
!pip install trimesh

In [ ]:
import csv, sys, os, shutil
from pathlib import Path
os.chdir('/content/MeshVTON'); sys.path.insert(0, '/content/MeshVTON')

# Eski gri/önden renderları temizle (FORCE_REGEN cell 15'ten gelir)
if globals().get('FORCE_REGEN', False):
    for d in ['data/processed/renders_3d', 'data/processed/normal_maps', 'data/processed/depth_maps']:
        if Path(d).exists(): shutil.rmtree(d)
    print('🗑️ eski renderlar silindi (dokulu + poza göre yeniden üretilecek)')

have_smplx = {f.stem for f in Path('data/processed/smplx_params').glob('*.npz')}
have_mesh  = {o.parent.name for o in Path('data/garments_3d').rglob('*.obj')}
data = list(csv.reader(open('data/raw/train_pairs.csv')))[1:]
valid = [(p, g) for p, g in data if p in have_smplx and g in have_mesh]

# Dengeli set: 5000 çift (~4 dk render, ~1 saat eğitim). Hepsini istersen valid[:] yap.
subset = valid[:5000]
with open('data/raw/train_pairs_valid.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['person_id', 'garment_id']); w.writerows(subset)
print(f"render edilecek: {len(subset)} çift (toplam geçerli: {len(valid)})")

for m in list(sys.modules):
    if m.startswith('src'): del sys.modules[m]

from src.data.preprocessing.render_garment import render_garments
render_garments(
    garments_dir='data/garments_3d', smplx_params_dir='data/processed/smplx_params',
    output_dir='data/processed/renders_3d', pairs_csv='data/raw/train_pairs_valid.csv',
    normal_maps_dir='data/processed/normal_maps', depth_maps_dir='data/processed/depth_maps',
    resolution=512, device='cuda')
print("render dosya:", len(list(Path('data/processed/renders_3d').glob('*.png'))))

---
## 7️⃣ Start Training 🚀

In [15]:
!cd /content/MeshVTON && python scripts/train_meshvton.py \
    --data_root data --pairs data/raw/train_pairs_valid.csv \
    --subset 1.0 --batch_size 4 --epochs 5 --lr 1e-4 \
    --height 512 --width 384 --save_dir checkpoints/meshvton

Loading frozen IDM-VTON pipeline...
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  t

---
## 8️⃣ Save checkpoints to Drive

In [14]:
import shutil
for fld in ['renders_3d','normal_maps','depth_maps']:
    shutil.make_archive(f'/content/drive/MyDrive/MeshVTON/{fld}', 'zip', f'data/processed/{fld}')
print("✅ render'lar Drive'a kaydedildi")

✅ render'lar Drive'a kaydedildi
